In [20]:
!pip install highspy -q

In [21]:
import os
import re
import glob
import time
import numpy as np
import pandas as pd
import highspy
from IPython.display import display

def ulsr_highs(datafile, time_limit=3600.0):
    start_time = time.time()

    with open(datafile, 'r') as file:
        linhas = [a.strip() for a in file.readlines() if a.strip()]

    # 1. Leitura do Arquivo
    N = int(linhas[0])
    FR_val = float(linhas[1]) # Setup Remanufatura
    F_val  = float(linhas[2]) # Setup Produção
    HR_val = float(linhas[3]) # Estoque Remanufatura
    H_val  = float(linhas[4]) # Estoque Produção

    D = np.array([float(v) for v in linhas[5].split()], dtype=float)
    R = np.array([float(v) for v in linhas[6].split()], dtype=float)

    # Custos
    P  = np.zeros(N)
    PR = np.zeros(N)
    F  = np.full(N, F_val)
    FR = np.full(N, FR_val)
    H  = np.full(N, H_val)
    HR = np.full(N, HR_val)

    # Instância do HiGHS
    h = highspy.Highs()

    # --- TUNAGEM DE ALTO DESEMPENHO ---
    h.setOptionValue("output_flag", False)
    h.setOptionValue("time_limit", float(time_limit))
    h.setOptionValue("presolve", "on")                 # Presolve ativo para simplificar o modelo
    h.setOptionValue("mip_rel_gap", 0.0001)            # Gap otimizado (0.01%)
    h.setOptionValue("mip_cutting_effort", 1.0)        # REATIVADO: Cortes são essenciais para Lot-Sizing
    h.setOptionValue("threads", 0)                     # 0 utiliza todos os núcleos de CPU disponíveis

    inf = highspy.kHighsInf

    # Mapeamento de Índices das Variáveis
    idx_x  = list(range(0, N))
    idx_s  = list(range(N, 2*N))
    idx_y  = list(range(2*N, 3*N))
    idx_xr = list(range(3*N, 4*N))
    idx_sr = list(range(4*N, 5*N))
    idx_yr = list(range(5*N, 6*N))

    # Criar Variáveis
    for i in range(N): h.addCol(P[i], 0.0, inf, 0, [], [])      # x
    for i in range(N): h.addCol(H[i], 0.0, inf, 0, [], [])      # s
    for i in range(N): h.addCol(F[i], 0.0, 1.0, 0, [], [])      # y
    for i in range(N): h.addCol(PR[i], 0.0, inf, 0, [], [])     # xr
    for i in range(N): h.addCol(HR[i], 0.0, inf, 0, [], [])     # sr
    for i in range(N): h.addCol(FR[i], 0.0, 1.0, 0, [], [])     # yr

    # Definir Integrabilidade (Binárias para Setup)
    for i in range(N):
        h.changeColIntegrality(idx_y[i], highspy.HighsVarType.kInteger)
        h.changeColIntegrality(idx_yr[i], highspy.HighsVarType.kInteger)

    # Pre-cálculo das somas acumuladas para acelerar o Big-M
    D_cum = np.cumsum(D[::-1])[::-1]  # D_cum[i] = sum(D[i:])
    R_cum = np.cumsum(R)              # R_cum[i] = sum(R[:i+1])

    # --- RESTRIÇÕES ---
    # 1. Balanço de Produtos Acabados
    h.addRow(D[0], D[0], 3, [idx_x[0], idx_xr[0], idx_s[0]], [1.0, 1.0, -1.0])
    for i in range(1, N):
        h.addRow(D[i], D[i], 4, [idx_s[i-1], idx_x[i], idx_xr[i], idx_s[i]], [1.0, 1.0, 1.0, -1.0])

    # 2. Balanço de Itens Retornados
    h.addRow(-R[0], -R[0], 2, [idx_xr[0], idx_sr[0]], [-1.0, -1.0])
    for i in range(1, N):
        h.addRow(-R[i], -R[i], 3, [idx_sr[i-1], idx_xr[i], idx_sr[i]], [1.0, -1.0, -1.0])

    # 3. Setup Produção: x[i] - (sum D)*y[i] <= 0
    for i in range(N):
        h.addRow(-inf, 0.0, 2, [idx_x[i], idx_y[i]], [1.0, -float(D_cum[i])])

    # 4. Setup Remanufatura: xr[i] - min(sum D, sum R)*yr[i] <= 0
    for i in range(N):
        cap_max = min(float(D_cum[i]), float(R_cum[i]))
        h.addRow(-inf, 0.0, 2, [idx_xr[i], idx_yr[i]], [1.0, -cap_max])

    # 5. Estoque Final Zero: s[N-1] == 0
    h.addRow(0.0, 0.0, 1, [idx_s[N-1]], [1.0])

    # Resolver Modelo
    h.run()

    exec_time = time.time() - start_time
    model_status = h.getModelStatus()
    info = h.getInfo()

    if model_status == highspy.HighsModelStatus.kOptimal:
        status_str = "Optimal"
        obj_val = info.objective_function_value
        bound_val = getattr(info, 'mip_dual_bound', obj_val)
        gap_val = 0.0
    elif model_status == highspy.HighsModelStatus.kTimeLimit:
        status_str = "TimeLimit"
        # Verifica se um valor objetivo viável foi encontrado (menor que o infinito do HiGHS)
        has_solution = info.objective_function_value < inf
        obj_val = info.objective_function_value if has_solution else None
        bound_val = getattr(info, 'mip_dual_bound', None)
        gap_val = getattr(info, 'mip_rel_gap', None)
    else:
        status_str = str(model_status)
        obj_val = None
        bound_val = None
        gap_val = None

    return {
        'status': status_str,
        'valor_otimo': obj_val,
        'bound': bound_val,
        'gap': gap_val,
        'tempo': exec_time
    }

In [22]:
def chave_natural(caminho):
    """Extrai os números do nome do arquivo para ordenar 52_2 antes de 52_10."""
    nome = os.path.basename(caminho)
    partes = re.split(r'(\d+)', nome)
    return [int(p) if p.isdigit() else p for p in partes]

pasta = "/content/drive/MyDrive/Colab Notebooks/sifaleras"
extensao = "*.txt"

inicio = 97   # posição inicial
fim    = 108 # posição final (inclusive)

TIMEOUT_SECONDS = 3600

# Busca e ordena os arquivos
arquivos = sorted(glob.glob(os.path.join(pasta, extensao)), key=chave_natural)
arquivos = arquivos[inicio-1 : fim]

print(f"Rodando {len(arquivos)} instância(s) em ordem com limite de tempo de {TIMEOUT_SECONDS} segundos...")

resultados = []
for arq in arquivos:
    res = ulsr_highs(arq, time_limit=TIMEOUT_SECONDS)

    instancia   = os.path.basename(arq)
    objval      = res.get('valor_otimo')
    bound       = res.get('bound', objval)
    gap         = res.get('gap')
    tempo       = res.get('tempo', 0.0)
    otimalidade = res.get('status', 'N/A')

    resultados.append({
        'INSTANCIA': instancia,
        'BOUND': bound,
        'OBJVAL': objval,
        'GAP': gap,
        'TIME': tempo,
        'OTIMALIDADE': otimalidade
    })

# DataFrame com os dados
df = pd.DataFrame(resultados)

# Tabela formatada para exibição
tabela = pd.DataFrame({
    'INSTANCIA':   df['INSTANCIA'],
    'BOUND':       df['BOUND'].map(lambda v: f"{v:.2f}" if pd.notnull(v) and isinstance(v, (int, float)) else "N/A"),
    'OBJVAL':      df['OBJVAL'].map(lambda v: f"{v:.2f}" if pd.notnull(v) and isinstance(v, (int, float)) else "N/A"),
    'GAP':         df['GAP'].map(lambda v: f"{v*100:.3f}%" if pd.notnull(v) and isinstance(v, (int, float)) else "N/A"),
    'TIME':        df['TIME'].map(lambda v: f"{v:.4f}" if pd.notnull(v) and isinstance(v, (int, float)) else "N/A"),
    'OTIMALIDADE': df['OTIMALIDADE']
})

display(tabela)

Rodando 12 instância(s) em ordem com limite de tempo de 3600 segundos...


,INSTANCIA,BOUND,OBJVAL,GAP,TIME,OTIMALIDADE
0,52_97.txt,32949.67,32952.40,0.000%,9.9547,Optimal
1,52_98.txt,33328.88,33332.20,0.000%,15.7093,Optimal
2,52_99.txt,33069.71,33072.40,0.000%,12.9135,Optimal
3,52_100.txt,33112.09,33115.00,0.000%,13.8300,Optimal
4,52_101.txt,36282.39,36286.00,0.000%,26.5172,Optimal
5,52_102.txt,37201.88,37205.50,0.000%,40.4082,Optimal
6,52_103.txt,36170.07,36173.50,0.000%,13.4011,Optimal
7,52_104.txt,36813.37,36817.00,0.000%,58.5521,Optimal
8,52_105.txt,38619.02,38622.80,0.000%,25.4533,Optimal
9,52_106.txt,40306.44,40310.40,0.000%,96.7845,Optimal


In [23]:
# CÉLULA 2: Salvar Tabela na Pasta de Trabalho

# Caminhos para salvar
caminho_csv = os.path.join("/content/drive/MyDrive/Colab Notebooks/", "resultado_milp_highs9.csv")
caminho_excel = os.path.join("/content/drive/MyDrive/Colab Notebooks/", "resultado_milp_highs9.xlsx")

# Salva em CSV (encoding utf-8-sig evita problemas de acentuação no Excel)
tabela.to_csv(caminho_csv, index=False, encoding='utf-8-sig')

# Salva em Excel (.xlsx)
tabela.to_excel(caminho_excel, index=False)

print(f"Tabela salva com sucesso na pasta:\n  - CSV: {caminho_csv}\n  - Excel: {caminho_excel}")

Tabela salva com sucesso na pasta:
  - CSV: /content/drive/MyDrive/Colab Notebooks/resultado_milp_highs9.csv
  - Excel: /content/drive/MyDrive/Colab Notebooks/resultado_milp_highs9.xlsx
